# CrossCodeEval adapter demo

Runs **one real CrossCodeEval example** through the full, unmodified pipeline:
loads the example (task_id/repository/file/prompt/groundtruth), clones and indexes
the referenced GitHub repo on demand, nominates candidates (BM25 + symbol +
dependency), lets Qwen pick useful ones, then StarCoder generates the completion.

Only `prompt` and `file` are ever passed to retrieval/selection/generation --
`groundtruth`/`right_context` are extracted but never fed downstream.

Uses the Hugging Face backends (not Ollama) -- proven more reliable on Colab.

**Before running**: Runtime -> Change runtime type -> select a GPU (T4 is fine).

In [ ]:
# Confirm a GPU is actually visible to this session -- if this errors or
# shows no GPU, Runtime -> Manage sessions -> terminate all, then reconnect.
!nvidia-smi

## 1. Get the project code

In [ ]:
%cd /content
import shutil, os
if os.path.exists("repo-code-completion"):
    shutil.rmtree("repo-code-completion")
!git clone "https://github.com/Robertkiza0/repo-code.git" repo-code-completion
%cd /content/repo-code-completion
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -q tree-sitter tree-sitter-python tree-sitter-java tree-sitter-typescript tree-sitter-c-sharp rank-bm25 requests
!pip install -q torch transformers accelerate bitsandbytes

## 3. Hugging Face login

Add your token as a Colab Secret named `HF_TOKEN` (padlock icon, left sidebar)
before running this -- never pasted directly into the notebook.

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()  # prompts for the token interactively (input is hidden)

## 4. Run one CrossCodeEval example through the full pipeline

First run clones + indexes the referenced repo (cached under `data/cceval/` for
later runs). `-n` picks a different example index (0-19) from the 20-example
sample; pass `--jsonl data/python/line_completion.jsonl` instead to draw from
the full dataset (requires the full CrossCodeEval archive to be extracted --
see the main README).

In [ ]:
import time
from evaluation.cceval_adapter import load_cceval_example, locate_repo_index, print_result, run_one_example
from selection.backends import HuggingFaceBackend
from selection.llm_selector import LLMSelector
from generation.backends import HuggingFaceGenerationBackend
from generation.generator import CompletionGenerator

EXAMPLE_INDEX = 0

example = load_cceval_example(index=EXAMPLE_INDEX)
chunks = locate_repo_index(example["repository"])  # clones + indexes on first run, cached after

selector = LLMSelector(chunks, backend=HuggingFaceBackend())  # Qwen2.5-Coder-7B-Instruct, 4-bit
generator = CompletionGenerator(chunks, backend=HuggingFaceGenerationBackend())  # StarCoder2-3b, 4-bit

t0 = time.time()
result = run_one_example(index=EXAMPLE_INDEX, selector=selector, generator=generator)
print(f"took {time.time() - t0:.1f}s\n")

print_result(chunks, result)